In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import freqz
from ipywidgets import Dropdown, IntSlider, FloatSlider, HBox, VBox, HTML, Layout, interactive_output
from IPython.display import display

# ============================================================
# FREQUENCY SAMPLING METHOD — INTERACTIVE DEMONSTRATION
# ============================================================

plt.rcParams.update({'font.size':11.5,'axes.titlesize':13,'axes.labelsize':11.5,'xtick.labelsize':10.5,'ytick.labelsize':10.5,'legend.fontsize':10})

# ============================================================
# DOCUMENTATION
# ============================================================

documentation = HTML("""
<style>

.fs-root{
    width:970px;
    max-width:970px;
    font-family:Arial,sans-serif;
}

.fs-header{
    background:linear-gradient(90deg,#176b87,#2399b5);
    color:white;
    padding:10px 15px;
    border-radius:8px 8px 0 0;
    font-size:18px;
    font-weight:bold;
}

.fs-doc{
    background:#f4fbfd;
    border:1px solid #b7dce4;
    border-top:none;
    padding:10px 13px;
    border-radius:0 0 8px 8px;
    font-size:13.5px;
    line-height:1.55;
    margin-bottom:9px;
}

.fs-title{
    font-weight:bold;
    color:#176b87;
    font-size:14px;
    margin-bottom:6px;
}

.fs-info{
    width:944px;
    border:1px solid #b7dce4;
    border-radius:7px;
    padding:9px 12px;
    font-size:13.5px;
    line-height:1.5;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea{
    overflow-x:visible !important;
    max-width:none !important;
}

</style>

<div class="fs-root">

<div class="fs-header">
Frequency Sampling Method — Interactive Demonstration
</div>

<div class="fs-doc">

<b>Purpose.</b>
This notebook demonstrates the basic principle of the frequency sampling method for FIR-filter design.
Samples of the desired frequency response are specified at the equally spaced frequencies
ω<sub>k</sub>=2πk/N, and the corresponding FIR impulse response is obtained through the inverse
discrete Fourier transform.

<br><br>

<b>What the controls show.</b>
The example considers a real, symmetric low-pass FIR filter with generalized linear phase.
The parameter <b>N</b> determines the number of frequency samples and FIR coefficients.
<b>Transition k</b> selects the frequency sample used as the transition sample, while
<b>Transition value</b> specifies the magnitude assigned to that sample.

<br><br>

<b>What to observe.</b>
A sharp sampled transition such as {1,1,1,1,0,0,...} generally produces stronger oscillations.
Assigning an intermediate value such as 0.4, 0.5 or 0.7 to the transition sample changes the
resulting FIR response and can improve the stopband behavior.

</div>

</div>
""")

# ============================================================
# CONTROLS
# ============================================================

N_control = Dropdown(options=[9,11,13,15,17,19,21,23,25,27,29,31],value=15,description='N:',style={'description_width':'30px'},layout=Layout(width='180px'))

k_control = IntSlider(value=4,min=1,max=7,step=1,description='Transition k:',continuous_update=True,style={'description_width':'90px'},layout=Layout(width='300px'))

T_control = FloatSlider(value=0.40,min=0,max=1,step=0.05,description='Transition value:',continuous_update=True,readout_format='.2f',style={'description_width':'110px'},layout=Layout(width='300px'))

controls = VBox([
    HTML('<div class="fs-title">Interactive controls</div>'),
    HBox([N_control,k_control,T_control],layout=Layout(width='930px',justify_content='space-between',align_items='center'))
],layout=Layout(width='970px',border='1px solid #b7dce4',padding='9px 12px',margin='0 0 8px 0'))

info = HTML(layout=Layout(width='970px',margin='0 0 8px 0'))

# ============================================================
# FREQUENCY-SAMPLING DESIGN
# ============================================================

def calculate_design(N,k_transition,T):
    M = (N-1)//2
    k = np.arange(N)

    A = np.zeros(N)
    A[:k_transition] = 1.0
    A[k_transition] = T

    for ki in range(1,M+1):
        A[N-ki] = A[ki]

    omega_k = 2*np.pi*k/N
    Hk = A*np.exp(-1j*omega_k*M)
    h = np.real_if_close(np.fft.ifft(Hk),tol=1000).real

    omega,H = freqz(h,worN=8192)
    Hmag = np.abs(H)
    Hdb = 20*np.log10(np.maximum(Hmag,1e-6))

    return M,A,h,omega,Hmag,Hdb

# ============================================================
# INTERACTIVE PLOT FUNCTION
# ============================================================

def draw_frequency_sampling(N,k_transition,T):
    M,A,h,omega,Hmag,Hdb = calculate_design(N,k_transition,T)

    L = (N-1)//2
    kp = np.arange(L+1)
    omega_samples = 2*kp/N
    omega_transition = 2*k_transition/N

    info.value = f"""
    <div class="fs-info">

    <div class="fs-title">Current frequency-sampling design</div>

    <div style="display:flex;gap:30px;">

        <div style="flex:1;">
        Number of frequency samples: <b>N = {N}</b><br>
        FIR length: <b>{N} samples</b><br>
        Linear-phase delay: <b>(N−1)/2 = {M} samples</b>
        </div>

        <div style="flex:1;">
        Transition sample index: <b>k = {k_transition}</b><br>
        Transition-sample frequency: <b>ω<sub>{k_transition}</sub> = {omega_transition:.3f}π</b>
        </div>

        <div style="flex:1;">
        Transition-sample magnitude: <b>|H<sub>d</sub>[{k_transition}]| = {T:.2f}</b><br>
        Frequency-sample spacing: <b>Δω = {2/N:.3f}π</b>
        </div>

    </div>

    </div>
    """

    fig,axes = plt.subplots(2,2,figsize=(10.8,7.2))
    ax1,ax2,ax3,ax4 = axes.flat

    # --------------------------------------------------------
    # 1. DESIRED FREQUENCY SAMPLES
    # --------------------------------------------------------

    markerline,stemlines,baseline = ax1.stem(omega_samples,A[:L+1],linefmt='r-',markerfmt='ro',basefmt=' ')
    plt.setp(markerline,markersize=5)
    plt.setp(stemlines,linewidth=1.3)

    ax1.axvline(omega_transition,linestyle='--',linewidth=1.0,label='Transition sample')
    ax1.set_xlim(-0.03,1.03)
    ax1.set_ylim(-0.08,1.12)
    ax1.set_title('Samples of the Desired Frequency Response')
    ax1.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax1.set_ylabel(r'$|H_d[k]|$')
    ax1.grid(True,linestyle=':',alpha=0.25)
    ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),frameon=False)

    # --------------------------------------------------------
    # 2. RESULTING MAGNITUDE RESPONSE
    # --------------------------------------------------------

    ax2.plot(omega/np.pi,Hmag,color='red',linewidth=1.6,label='Resulting FIR response')
    ax2.plot(omega_samples,A[:L+1],'o',markersize=5,label='Specified samples')

    ax2.set_xlim(0,1)
    ax2.set_ylim(-0.08,1.15)
    ax2.set_title('Resulting Magnitude Response')
    ax2.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax2.set_ylabel(r'$|H(e^{j\omega})|$')
    ax2.grid(True,linestyle=':',alpha=0.25)
    ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),ncol=2,frameon=False)

    # --------------------------------------------------------
    # 3. FIR IMPULSE RESPONSE
    # --------------------------------------------------------

    n = np.arange(N)

    markerline,stemlines,baseline = ax3.stem(n,h,linefmt='r-',markerfmt='ro',basefmt=' ')
    plt.setp(markerline,markersize=4)
    plt.setp(stemlines,linewidth=1.1)

    ax3.axvline(M,linestyle='--',linewidth=1.0,label=f'Delay = {M}')
    ax3.set_title('FIR Impulse Response from the IDFT')
    ax3.set_xlabel('Sample index $n$')
    ax3.set_ylabel('$h[n]$')
    ax3.grid(True,linestyle=':',alpha=0.25)
    ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.16),frameon=False)

    # --------------------------------------------------------
    # 4. MAGNITUDE RESPONSE IN dB
    # --------------------------------------------------------

    ax4.plot(omega/np.pi,Hdb,color='red',linewidth=1.5)

    ax4.set_xlim(0,1)
    ax4.set_ylim(-100,5)
    ax4.set_title('Magnitude Response in dB')
    ax4.set_xlabel(r'Normalized frequency $\omega/\pi$')
    ax4.set_ylabel('Magnitude [dB]')
    ax4.grid(True,linestyle=':',alpha=0.25)

    plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.11,wspace=0.29,hspace=0.40)
    plt.show()
    plt.close(fig)

# ============================================================
# DYNAMIC RANGE OF TRANSITION k
# ============================================================

def update_k_range(change):
    L = (N_control.value-1)//2
    k_control.max = L

    if k_control.value > L:
        k_control.value = L

N_control.observe(update_k_range,names='value')

# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

plots = interactive_output(draw_frequency_sampling,{'N':N_control,'k_transition':k_control,'T':T_control})
plots.layout = Layout(width='970px',overflow='visible')

# ============================================================
# DISPLAY
# ============================================================

display(documentation)
display(controls)
display(info)
display(plots)